In [1]:
import json
from datasets import load_dataset

In [2]:
datasets = load_dataset('hate-speech-portuguese/hate_speech_portuguese', split='train[:10%]')


In [3]:
print(datasets)

Dataset({
    features: ['text', 'label', 'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'],
    num_rows: 567
})


In [6]:
datasets = datasets.remove_columns([
    'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'
])

In [7]:
print(datasets)

Dataset({
    features: ['text', 'label'],
    num_rows: 567
})


In [15]:
datasets = datasets.train_test_split(test_size=0.2)

In [16]:
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 453
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 114
    })
})


In [17]:
datasets["train"]["text"]

['@angelicamorango NUsssss, magavilha, heinnn. MAssa esse promo!',
 'a cota de negro aumentou nesse BBB né',
 'Apresento-vos o meu missil, AEN17.\nTotalmente dirigida para Angola sem dó e piedade! Se Angola não é nossa, mais va _ https://t.co/1eA3Rzkjej',
 'Ah, o perigo do Islão, e tal, e a civilização europeia, e não sei quê. https://t.co/GBJPgWqhYO',
 'Aposto que um bom advogado podia criar um bom argumento do fato do presidente Obama grampear meus telefones em Outu _ https://t.co/VOacDfTDaQ',
 '@ajulysantos que não é caso da brasileira..porque com 5 kg de cocaína..é o peso de um bulldogue frances',
 'A greve dos Estivadores\nhttps://t.co/EcYwmGj8Aq#RenovarPortugal #PNR',
 'A mídia de notícias falsas está insana com suas teorias conspiratórias e ódio cego. @MSNBC & @CNN são inassistíveis _ https://t.co/biN6YswUWW',
 'A Luisa é pesquisadora e lançou esse projeto para apoiar um novo escritor. Ajude com qualquer quantia! Veja mais... https://t.co/22oWkyfG9w',
 '@ajulysantos isso! A mulh

In [21]:
def removeN(example):
    example['text'] = example['text'].replace("\n", " ")
    return example

In [22]:
datasets = datasets.map(removeN)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [23]:
datasets['train'][3]

{'text': 'Ah, o perigo do Islão, e tal, e a civilização europeia, e não sei quê. https://t.co/GBJPgWqhYO',
 'label': 0}

In [25]:
# label 0 -> No hate speech
# label 1 -> Hate speech

def labelChange(example):
    example["label_text"] = 'No Hate Speech' if example['label'] == 0 else 'Hate Speech'
    return example


In [26]:
datasets = datasets.map(labelChange)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [ ]:
datasets = datasets.remove_columns(['label'])

In [29]:
print(datasets['train'][0])

{'text': '@angelicamorango NUsssss, magavilha, heinnn. MAssa esse promo!', 'label_text': 'No Hate Speech'}


In [ ]:
# CONSTRUCAO DO OBJETO PARA OPENAI

def dataset_to_jsonl(dataset, file_name):
    with open(file_name, 'w', encoding='utf-8') as f:
        for example in dataset:
            json_obj = {"messages": [
                {"role": "system", "content": "Seu trabalho é classificar os comentários do usuário em Hate Speech e No Hate Speech."},
                {"role": "user", "content": example['text']},
                {"role": "assistant", "content": example['label_text']},
            ]}
            f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [31]:
dataset_to_jsonl(datasets['train'], 'train.jsonl')

In [32]:
dataset_to_jsonl(datasets['test'], 'validation.jsonl')

In [34]:
from openai import OpenAI
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [36]:
client = OpenAI()

In [37]:
client.files.create(
    file=open("train.jsonl", 'rb'),
    purpose="fine-tune"
)

FileObject(id='file-5Z8ugesCePd1b8WV9pDNC7', bytes=146246, created_at=1778343311, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [38]:
client.files.create(
    file=open("validation.jsonl", 'rb'),
    purpose="fine-tune"
)

FileObject(id='file-BoA2Thq2W73rpuyvD1bWtP', bytes=37576, created_at=1778343321, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [ ]:
# Deprecated

client.fine_tuning.jobs.create(
    training_file='file-5Z8ugesCePd1b8WV9pDNC7',
    validation_file='file-BoA2Thq2W73rpuyvD1bWtP',
    model='gpt-3.5-turbo'
)